# SHAP (SHapley Additive exPlanations) - Interactive Tutorial

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alan-turing-institute/tea-techniques/blob/main/tutorials/notebooks/shap/shap-tutorial.ipynb)

**Assurance Goals:** Explainability, Fairness, Reliability
**Difficulty:** Intermediate
**Time:** ~60-90 minutes

---

## 1. Overview

### What is SHAP?

SHAP (SHapley Additive exPlanations) is a technique that explains individual predictions by computing how much each feature contributed to that prediction. It's based on **Shapley values** from cooperative game theory—a mathematically principled way to fairly distribute credit among team members.

You can think of it as follows: if a team wins a prize, how do you fairly divide it based on each member's contribution? Shapley values solve this problem by considering all possible team combinations and measuring each member's marginal contribution.

### The "Before and After" Transformation

Without SHAP:
```
Input: [8 features about a house] -> Model -> Output: $354,000
```
*Why this prediction? We have no idea.*

With SHAP:
```
Input: [8 features] -> Model -> Output: $354,000
                                      |
                             SHAP Decomposition:
                             - Median Income: +$82,000
                             - House Age: -$15,000
                             - Avg Rooms: +$23,000
                             - Location: +$45,000
                             - ... (other features)
                             - Base value: $219,000
                             --------------------
                             = $354,000
```
*Now we can begin to explain exactly why.*

### When should I use SHAP?

- **Model debugging**: "Why is my model making this unexpected prediction?"
- **Regulatory compliance**: "We need to explain decisions to regulators."
- **Stakeholder communication**: "Why was this loan denied?"
- **Feature importance**: "Which features matter most, globally and locally?"
- **Explainability assurance cases**: "We need evidence that our model is explainable."

### When should I NOT use SHAP?

- **Real-time inference**: SHAP computation adds latency; consider pre-computing explanations for production systems
- **Very high-dimensional data**: With hundreds of features, SHAP plots become unreadable and computation is expensive
- **When a simpler method suffices**: If your model is inherently interpretable (e.g. logistic regression with few features), SHAP may be overkill
- **Causal claims**: SHAP shows associations, not causes. If you need causal explanations, consider causal inference methods

### Key Concepts You'll Learn in This Tutorial

- **SHAP values**: How much each feature contributes to a prediction (can be positive or negative)
- **Base value**: The average prediction across your training data - SHAP values are *relative* to this
- **Local vs Global**: SHAP provides both individual and aggregate explanations
- **Explainer types**: Different algorithms optimised for different model types

## 2. Prerequisites

### Required Knowledge

- Basic Python programming
- Familiarity with scikit-learn style ML workflow (fit, predict)
- Understanding of what features and predictions are

### Setup

Run the cell below to install required packages:

In [ ]:
# Install required packages (auto-detects Colab/Kaggle environments)
import subprocess, sys

def is_colab():
    try:
        import google.colab
        return True
    except ImportError:
        return False

def is_kaggle():
    import os
    return 'KAGGLE_KERNEL_RUN_TYPE' in os.environ

if is_colab() or is_kaggle():
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '-q',
        'shap>=0.42.0,<1.0.0', 'xgboost>=1.5.0,<3.0.0',
        'scikit-learn>=1.0.0,<2.0.0', 'matplotlib>=3.4.0,<4.0.0',
        'pandas>=1.3.0,<3.0.0'
    ])
    print("Packages installed for cloud environment.")
else:
    print("Local environment detected. Ensure you've run: pip install -r requirements.txt")

In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap
import xgboost as xgb
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import json
import hashlib
from datetime import datetime
import warnings

# Targeted warning suppression (avoid blanket ignore)
warnings.filterwarnings('ignore', category=FutureWarning, module='shap')
warnings.filterwarnings('ignore', category=UserWarning, module='shap')

# Set random seed for reproducibility
np.random.seed(42)

# Initialise SHAP JS visualisations for notebooks
# Note: If plots don't render, try using matplotlib=True in plot commands
# or ensure you're using a compatible Jupyter environment
try:
    shap.initjs()
except Exception:
    print("Note: shap.initjs() failed - some interactive plots may not render.")
    print("Static matplotlib plots will still work.")

print(f"SHAP version: {shap.__version__}")
print(f"XGBoost version: {xgb.__version__}")
print("Setup complete!")

### Dataset: California Housing

We'll use the California Housing dataset - a classic ML dataset where we predict median house values based on features like:
- **MedInc**: Median income in the block group
- **HouseAge**: Median age of houses in the block
- **AveRooms**: Average number of rooms per household
- **AveBedrms**: Average number of bedrooms per household
- **Population**: Block group population
- **AveOccup**: Average number of household members
- **Latitude/Longitude**: Location coordinates

**Why this dataset?** House prices are intuitive - we all have intuitions about what makes a house expensive. This makes it easier to validate whether SHAP explanations make sense.

In [ ]:
# Load the California Housing dataset
housing = fetch_california_housing()
X = pd.DataFrame(housing.data, columns=housing.feature_names)
y = housing.target  # Median house value in $100,000s

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training samples: {len(X_train):,}")
print(f"Test samples: {len(X_test):,}")
print(f"\nFeatures: {list(X.columns)}")
print(f"\nTarget: Median house value (in $100,000s)")
print(f"  Mean: ${y.mean() * 100000:,.0f}")
print(f"  Range: ${y.min() * 100000:,.0f} - ${y.max() * 100000:,.0f}")

In [ ]:
# Quick look at the data
X_train.head()

## 3. How It Works

### Intuition: The Coalition Game

Imagine you're a project manager and your team just won a £1,000 bonus. The team members are:
- Alice (data analyst)
- Bob (developer)
- Carol (designer)

How do you fairly divide the bonus based on each person's contribution?

**The Shapley approach:**
1. Consider every possible team combination
2. For each combination, measure the marginal contribution when each person joins
3. Average each person's marginal contributions across all orderings

This gives a mathematically "fair" allocation that satisfies four key axioms:
- **Null player**: If you contribute nothing, you get nothing
- **Symmetry**: If two people contribute equally, they get equal shares
- **Efficiency**: The shares sum to the total prize
- **Linearity (Additivity)**: Shapley values for a combined game equal the sum of Shapley values for each game separately

### SHAP Workflow Overview

```
                    +-------------------+
                    |   Trained Model   |
                    +--------+----------+
                             |
                    +--------v----------+
                    |  SHAP Explainer   |
                    | (Tree / Kernel /  |
                    |  Linear / Deep)   |
                    +--------+----------+
                             |
              +--------------+--------------+
              |                             |
    +---------v---------+       +-----------v-----------+
    | Local Explanations|       | Global Explanations   |
    | (per prediction)  |       | (across all samples)  |
    |                   |       |                       |
    | - Waterfall plot  |       | - Beeswarm plot       |
    | - Force plot      |       | - Bar plot            |
    +--------+----------+       | - Dependence plots    |
             |                  +-----------+-----------+
             |                              |
             +-------------+----------------+
                           |
                  +--------v---------+
                  | Assurance Evidence|
                  | - Stability tests |
                  | - Evidence JSON   |
                  | - Limitations doc |
                  +------------------+
```

### Interactive Example: Manual Shapley Calculation

Let's calculate Shapley values by hand for a simple 3-feature prediction.

Suppose we have a tiny model that predicts house price using just 3 features:
- **A** = Income level
- **B** = House size
- **C** = Location score

And we can measure how the prediction changes with different feature combinations:

**A note on currency:** The example below uses British pounds (£) because it's a standalone game-theory analogy — not real data. When we move to the California Housing dataset in Section 4, all values will be in US dollars ($), which is the dataset's native currency.

In [ ]:
# Simulated predictions for different feature subsets
# (In practice, "missing" features use average/baseline values)

coalition_values = {
    frozenset():        200,   # No features (baseline)
    frozenset(['A']):   280,   # Only Income
    frozenset(['B']):   230,   # Only Size
    frozenset(['C']):   220,   # Only Location
    frozenset(['A','B']): 320, # Income + Size
    frozenset(['A','C']): 310, # Income + Location
    frozenset(['B','C']): 260, # Size + Location
    frozenset(['A','B','C']): 350,  # All features (full prediction)
}

print("Predictions with different feature combinations (in £1000s):")
for features, pred in coalition_values.items():
    feature_str = ', '.join(sorted(features)) if features else '(none)'
    print(f"  {feature_str:15} -> £{pred}k")

In [ ]:
# Calculate Shapley value for feature A (Income)
# We enumerate ALL 3! = 6 orderings and average A's marginal contribution

from itertools import permutations

features = ['A', 'B', 'C']

def marginal_contribution(feature, coalition, coalition_values):
    """Calculate marginal contribution of adding 'feature' to 'coalition'"""
    with_feature = coalition_values[frozenset(coalition | {feature})]
    without_feature = coalition_values[frozenset(coalition)]
    return with_feature - without_feature

print("Calculating Shapley value for feature A (Income):\n")
print(f"{'Ordering':<12} {'Coalition before A':<20} {'Marginal Contribution':>22}")
print("-" * 58)

marginals = []
for perm in permutations(features):
    # Coalition = features that appear before A in this ordering
    a_pos = list(perm).index('A')
    coalition = set(perm[:a_pos])
    mc = marginal_contribution('A', coalition, coalition_values)
    marginals.append(mc)
    coalition_str = ', '.join(sorted(coalition)) if coalition else '(empty)'
    print(f"{''.join(perm):<12} {coalition_str:<20} £{mc:>+4}k")

shapley_A = np.mean(marginals)
print(f"\nShapley value for A = average of {len(marginals)} orderings = £{shapley_A:.2f}k")

In [ ]:
# Calculate all Shapley values using the weighted coalition formula
from itertools import combinations
from math import factorial

def calculate_shapley(feature, all_features, coalition_values):
    """Calculate exact Shapley value for a feature"""
    other_features = [f for f in all_features if f != feature]
    n = len(all_features)
    shapley_value = 0

    # Iterate over all possible coalitions (subsets of other features)
    for size in range(len(other_features) + 1):
        for coalition in combinations(other_features, size):
            coalition_set = set(coalition)
            mc = marginal_contribution(feature, coalition_set, coalition_values)
            # Weight by coalition size
            weight = factorial(len(coalition_set)) * factorial(n - len(coalition_set) - 1) / factorial(n)
            shapley_value += weight * mc

    return shapley_value

print("Shapley Values (manual calculation):")
print("=" * 40)
shapley_values_manual = {}
for f in features:
    sv = calculate_shapley(f, features, coalition_values)
    shapley_values_manual[f] = sv
    print(f"  {f} (Income/Size/Location): £{sv:.1f}k")

print(f"\nBase value (no features): £{coalition_values[frozenset()]}k")
print(f"Sum of Shapley values: £{sum(shapley_values_manual.values()):.1f}k")
print(f"Full prediction: £{coalition_values[frozenset(features)]}k")
print(f"\nCheck: Base + Shapley values = {coalition_values[frozenset()]} + {sum(shapley_values_manual.values()):.1f} = {coalition_values[frozenset()] + sum(shapley_values_manual.values()):.1f}k")

### The Key Insight: Additivity

Notice that **base value + all SHAP values = full prediction**. This is guaranteed by the mathematics!

$$\text{Prediction} = \text{Base Value} + \sum_{i=1}^{n} \text{SHAP}_i$$

This "additivity" property means we can decompose *any* prediction into interpretable parts.

**Note:** The section below uses collapsible `<details>` HTML. If you are using a screen reader, the full content is accessible by activating the summary element.

<details>
<summary><b>Mathematical Foundations (click to expand)</b></summary>

### The Shapley Value Formula

For a feature $i$ with $n$ total features:

$$\phi_i = \sum_{S \subseteq N \setminus \{i\}} \frac{|S|! (n - |S| - 1)!}{n!} [f(S \cup \{i\}) - f(S)]$$

Where:
- $N$ is the set of all features
- $S$ is a subset of features not including $i$
- $f(S)$ is the model prediction using only features in $S$
- The fraction is a weighting factor based on coalition size

### Computational Challenge

For $n$ features, we need to evaluate $2^n$ feature subsets. With 20 features, that's over 1 million evaluations per prediction!

SHAP solves this with efficient approximations:
- **TreeSHAP**: Exact, polynomial-time algorithm for tree models
- **KernelSHAP**: Approximation via weighted linear regression for any model

</details>

### The Base Value: A Critical Concept

**SHAP values are always relative to the base value (expected prediction).**

This means:
- A SHAP value of +$50,000 means "this feature adds $50,000 *compared to the average house*"
- A SHAP value of -$10,000 means "this feature reduces the predicted value by $10,000 *from the average*"

In the California Housing dataset, the average house value is approximately $207,000. A house predicted at $150,000 might have:
- Small positive SHAP values (features slightly above average)
- Large negative SHAP values (features that push the price down)

**Common mistake**: Thinking a small negative SHAP value means the feature makes the house "cheap". It just means that feature contributes less than average - the house could still be expensive!

### Correlated Features

Correlated features present a challenge for SHAP. We discuss this in detail in **Section 4.2** after you've seen how TreeExplainer works and understand the difference between Interventional and Path-Dependent modes.

## 4. Hands-On Implementation

### 4.1 Train a Model

First, let's train an XGBoost regressor. We'll use this model for all our SHAP explanations.

In [ ]:
# Train XGBoost model
model = xgb.XGBRegressor(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    # n_jobs=-1 uses all CPU cores; results may vary slightly across machines
    # due to floating-point summation order. Set n_jobs=1 for exact reproducibility.
    n_jobs=-1
)

model.fit(X_train, y_train)

# Evaluate
y_pred = model.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("Model Performance:")
print(f"  RMSE: ${rmse * 100000:,.0f}")
print(f"  R²: {r2:.3f}")
print(f"\nThe model explains {r2*100:.1f}% of the variance in house prices.")

### 4.2 Basic Usage - TreeExplainer

For tree-based models (XGBoost, LightGBM, CatBoost, Random Forest), use `TreeExplainer` - it's exact and fast.

In [ ]:
# Create TreeExplainer
# We pass background data to establish the base value
# Using 500 samples for stability (larger backgrounds improve consistency; see Yuan et al. 2022)
background = shap.sample(X_train, 500, random_state=42)

# IMPORTANT: Passing data= forces "Interventional" TreeSHAP
# - This ASSUMES feature independence when marginalising
# - Without data=, XGBoost uses "Path-Dependent" TreeSHAP (respects correlations)
# - We use Interventional here for stability, but note this distinction for assurance!
explainer = shap.TreeExplainer(model, data=background)

print(f"Explainer created with {len(background)} background samples")
print(f"Base value (expected prediction): ${explainer.expected_value * 100000:,.0f}")
print(f"\nNote: Using Interventional TreeSHAP (assumes feature independence)")
print("For path-dependent explanations, omit the data= parameter.")

### How SHAP Handles Correlated Features (IMPORTANT!)

Now that you've seen TreeExplainer, a critical nuance: **different SHAP algorithms handle correlated features differently**.

**TreeSHAP (path-dependent, default for tree models when `data=` is omitted):**
- Respects the correlations captured by the tree structure
- When features are correlated, credit gets "split" between them
- Example: In this dataset, MedInc and AveOccup are correlated (r ~ -0.43). Credit gets shared between them.

**Interventional TreeSHAP (what we used above by passing `data=`):**
- Breaks feature dependencies by assuming independence when marginalising
- Can produce different attributions than path-dependent TreeSHAP
- More stable across runs, but may misattribute credit for correlated features

**What this means for assurance:**
- With correlated features, identifying the "true driver" is difficult
- Credit splitting is mathematically correct but may not match human intuition
- Always check feature correlations and document them in your evidence!
- We explore the actual correlations in this dataset in Section 4.4 (Pitfall 2)

In [ ]:
# Compute SHAP values for test set
# This may take a minute...
shap_values = explainer(X_test)

print(f"SHAP values computed for {len(X_test)} test samples")
print(f"Shape: {shap_values.values.shape}")
print(f"  - {shap_values.values.shape[0]} samples")
print(f"  - {shap_values.values.shape[1]} features")

In [ ]:
# Verify additivity: base_value + sum(shap_values) = prediction
idx = 0  # First test sample
base = shap_values.base_values[idx]
shap_sum = shap_values.values[idx].sum()
prediction = model.predict(X_test.iloc[[idx]])[0]
computed_total = base + shap_sum

print("Additivity Check (first test sample):")
print(f"  Base value:        ${base * 100000:>12,.0f}")
print(f"  Sum of SHAP:       ${shap_sum * 100000:>12,.0f}")
print(f"  ─────────────────────────────────")
print(f"  Total:             ${computed_total * 100000:>12,.0f}")
print(f"  Model prediction:  ${prediction * 100000:>12,.0f}")

# Use np.isclose for precise verification (handles floating-point errors)
residual = abs(computed_total - prediction)
is_match = np.isclose(computed_total, prediction, rtol=1e-5)
print(f"\n  Residual: {residual:.2e} (effectively zero)")
print(f"  ✓ Match verified with np.isclose(rtol=1e-5): {is_match}")

**Pause and reflect:** Why is the additivity property (base + SHAP values = prediction) important for assurance? What would happen if explanations didn't sum to the actual prediction?

### 4.3 Interpreting Results

#### Local Explanations: Explaining Individual Predictions

**Waterfall Plot**: Shows how each feature pushes the prediction from the base value.

In [ ]:
# Waterfall plot for a single prediction
sample_idx = 0

print(f"Explaining prediction for test sample {sample_idx}:")
print(f"  Actual value: ${y_test.iloc[sample_idx] * 100000:,.0f}")
print(f"  Predicted value: ${model.predict(X_test.iloc[[sample_idx]])[0] * 100000:,.0f}")
print("\nFeature values for this house:")
for col in X_test.columns:
    print(f"  {col}: {X_test.iloc[sample_idx][col]:.2f}")

**Predict before you run:** Look at the feature values above. Which features do you think will push this prediction *up* (above average)? Which will push it *down*? Write your prediction, then run the waterfall plot below to check.

In [ ]:
# Waterfall plot
# Reading the plot:
# - E[f(X)] at bottom: base value (average prediction)
# - f(x) at top: final prediction
# - Red bars: features that INCREASE the prediction
# - Blue bars: features that DECREASE the prediction
# - Bar length: magnitude of contribution

shap.plots.waterfall(shap_values[sample_idx], max_display=10, show=False)
plt.title("Waterfall Plot: How Features Contribute to This Prediction\n"
          "(Red = increases price, Blue = decreases price)", fontsize=10)
plt.tight_layout()
plt.show()

**Accessibility note:** SHAP's default red/blue colour palette may be difficult to distinguish for colour-blind users. You can override the colour map with `cmap=plt.cm.coolwarm` or `cmap=plt.cm.PiYG` in beeswarm and scatter plots. For precise values, always access the underlying numerical data via `shap_values.values[idx]` rather than relying solely on visual inspection.

**Reading the Waterfall Plot:**

- Start at the **bottom**: This is E[f(X)], the base value (~average prediction)
- Move **upward**: Each bar shows a feature's contribution
- **Red bars** push the prediction **higher** (positive contribution)
- **Blue bars** push the prediction **lower** (negative contribution)
- End at the **top**: f(x), the final prediction
- The **number** next to each feature is its actual value for this sample

#### Exercise: Explain an Expensive House

Let's find and explain one of the most expensive houses in our test set.

In [ ]:
# Find an expensive house
all_predictions = model.predict(X_test)
expensive_idx = np.argmax(all_predictions)

print(f"Most expensive predicted house (index {expensive_idx}):")
print(f"  Predicted: ${all_predictions[expensive_idx] * 100000:,.0f}")
print(f"  Actual: ${y_test.iloc[expensive_idx] * 100000:,.0f}")

shap.plots.waterfall(shap_values[expensive_idx], max_display=10, show=False)
plt.tight_layout()
plt.show()

**Question**: Looking at the waterfall plot above, which features contribute most to making this house expensive? Does this match your intuition about what makes houses valuable?

In [ ]:
# TODO: Find the cheapest predicted house and create a waterfall plot
# Hint: use np.argmin(all_predictions) to find the index

# --- Uncomment the solution below when ready ---
# cheap_idx = np.argmin(all_predictions)
# print(f"Cheapest predicted house (index {cheap_idx}):")
# print(f"  Predicted: ${all_predictions[cheap_idx] * 100000:,.0f}")
# print(f"  Actual: ${y_test.iloc[cheap_idx] * 100000:,.0f}")
# shap.plots.waterfall(shap_values[cheap_idx], max_display=10, show=False)
# plt.tight_layout()
# plt.show()

**Exercise: Explain a Cheap House**

Now find and explain one of the *cheapest* predicted houses. What features push the prediction down?

#### Global Explanations: Understanding Overall Model Behaviour

**Beeswarm Plot**: Shows the distribution of SHAP values for all samples.

**Predict before you run:** Before looking at the beeswarm plot below, which feature do you expect to be most important globally for predicting California house prices? Write down your prediction, then run the next cell to check.

In [ ]:
# Beeswarm plot
# Reading the plot:
# - Each dot is one sample
# - X-axis: SHAP value (how much this feature affects prediction)
# - Colour: feature value (red = high, blue = low)
# - Features sorted by importance (most important at top)

shap.plots.beeswarm(shap_values, max_display=10, show=False)
plt.title("Beeswarm Plot: Feature Importance Distribution\n"
          "(Each dot = one sample, Colour = feature value)", fontsize=10)
plt.tight_layout()
plt.show()

**Reading the Beeswarm Plot:**

- Each **dot** represents one sample
- **X-axis**: SHAP value (contribution to prediction)
- **Colour**: Feature value (red = high, blue = low)
- **Vertical stacking**: Density of samples at that SHAP value
- Features are **sorted by importance** (most impactful at top)

**What to look for:**
- MedInc (income): Red dots on right = high income → higher prices ✓
- Latitude: The relationship with location (latitude affects coastal proximity)
- HouseAge: Does old or new matter more for price?

In [ ]:
# Bar plot: Mean absolute SHAP values (global feature importance)
shap.plots.bar(shap_values, max_display=10, show=False)
plt.title("Global Feature Importance\n(Mean |SHAP value| across all samples)", fontsize=10)
plt.tight_layout()
plt.show()

**Reading the Bar Plot:**

- Each bar shows the **mean absolute SHAP value** for that feature across all test samples
- This measures the feature's average *impact magnitude* (ignoring direction)
- MedInc dominates because income is the strongest predictor of house prices in California
- Compared to the beeswarm plot, the bar plot loses directional and distributional information but provides a cleaner summary for stakeholders
- Use bar plots in reports and presentations; use beeswarm plots for deeper analysis

#### Feature Interactions: Dependence Plots

**Dependence plots** show how one feature's SHAP value varies with its value, coloured by an interacting feature.

In [ ]:
# Dependence plot for MedInc (most important feature)
# X-axis: Feature value
# Y-axis: SHAP value for that feature
# Colour: Automatically chosen interacting feature

shap.plots.scatter(shap_values[:, "MedInc"], color=shap_values[:, "AveOccup"], show=False)
plt.title("How Median Income Affects Predictions\n"
          "(Colour = Average Occupancy)", fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# TODO: Create a dependence plot for HouseAge
# Hint: use shap.plots.scatter with shap_values[:, "HouseAge"]

# --- Uncomment the solution below when ready ---
# shap.plots.scatter(shap_values[:, "HouseAge"], color=shap_values[:, "MedInc"], show=False)
# plt.title("How House Age Affects Predictions\n(Colour = Median Income)", fontsize=10)
# plt.tight_layout()
# plt.show()

**Exercise: Explore Another Feature**

Create a dependence plot for `HouseAge`. What pattern do you see? Does older or newer housing have a stronger effect on predicted prices?

**Reading the Dependence Plot:**

- **X-axis**: The actual feature value (e.g. median income in $10,000s)
- **Y-axis**: The SHAP value for that feature (how much it pushes the prediction up or down)
- **Colour**: An interacting feature (here, AveOccup). Colour variation at the same x-value indicates an interaction
- **Non-linear patterns**: The curve is not a straight line — SHAP captures the model's non-linear learned relationship
- **Vertical spread**: At any given income level, the spread shows how other features modulate the effect of income

### 4.4 Common Pitfalls

#### Pitfall 1: Small Background Dataset

Research by Yuan et al. (2022) shows SHAP explanations can be unstable with small background datasets.

In [ ]:
# Demonstrate instability with small background
print("Comparing SHAP values with different background sizes...\n")

sample_idx = 0
results = []

for bg_size in [50, 100, 500]:
    bg = shap.sample(X_train, bg_size, random_state=42)
    exp = shap.TreeExplainer(model, data=bg)
    sv = exp(X_test.iloc[[sample_idx]])

    # Get SHAP value for MedInc (most important feature)
    medinc_shap = sv.values[0][0]  # First feature is MedInc
    results.append((bg_size, medinc_shap))
    print(f"Background size {bg_size:3d}: MedInc SHAP = {medinc_shap:.4f}")

print(f"\nVariation: {max(r[1] for r in results) - min(r[1] for r in results):.4f}")
print("\nRecommendation: Use 500+ background samples for stability.")

#### Pitfall 2: Correlated Features

Let's check the correlations in our dataset:

In [ ]:
# Check feature correlations
corr_matrix = X_train.corr()

# Find highly correlated pairs (|r| > 0.5)
print("Highly correlated feature pairs (|r| > 0.5):")
print("=" * 50)
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        r = corr_matrix.iloc[i, j]
        if abs(r) > 0.5:
            print(f"  {corr_matrix.columns[i]:12} ↔ {corr_matrix.columns[j]:12}: r = {r:.2f}")

print("\nWhen features are correlated:")
print("  - SHAP credit gets 'split' between them")
print("  - Hard to identify the 'true driver'")
print("  - Document correlations in your assurance evidence!")

**Pause and reflect:** Given the correlations above, how might this affect SHAP-based assurance claims? If two correlated features share credit, could a stakeholder mistakenly conclude that neither is important?

#### Pitfall 3: Confusing Correlation with Causation

SHAP values show **associations**, not causal effects!

Example: Latitude has high SHAP importance. This doesn't mean "moving a house north will change its price." It means the model has learnt that latitude is associated with price (likely because it correlates with proximity to expensive coastal areas).

### 4.5 Testing Explanation Stability

This is **critical for assurance** - we need to know if our explanations are reliable.

In [ ]:
NEGLIGIBLE_THRESHOLD = 0.01  # Features with mean |SHAP| below this are "negligible"

def test_shap_stability(model, X_train, X_test, n_runs=3, background_size=500,
                        n_test_samples=100):
    """
    Test SHAP explanation stability across multiple runs with different
    background samples. This is a 'unit test for explainability'.
    """
    feature_names = X_test.columns.tolist()
    n_features = len(feature_names)

    # Use a reproducible random sample of the test set
    test_subset = X_test.sample(n=min(n_test_samples, len(X_test)), random_state=42)

    # Store SHAP values from each run
    all_shap_values = []

    print(f"Running {n_runs} SHAP computations with different backgrounds...")
    for run in range(n_runs):
        # Different random background each time
        bg = shap.sample(X_train, background_size, random_state=run*100)
        exp = shap.TreeExplainer(model, data=bg)
        sv = exp(test_subset)
        all_shap_values.append(sv.values)
        print(f"  Run {run+1}/{n_runs} complete")

    # Stack and compute statistics
    all_shap_values = np.stack(all_shap_values, axis=0)  # (n_runs, n_samples, n_features)

    # Compute mean and std across runs for each sample/feature
    mean_shap = np.mean(all_shap_values, axis=0)
    std_shap = np.std(all_shap_values, axis=0)

    # Compute stability metrics per feature (averaged across samples)
    feature_mean_abs_shap = np.mean(np.abs(mean_shap), axis=0)
    feature_mean_std = np.mean(std_shap, axis=0)

    # Coefficient of variation (relative stability)
    cv = feature_mean_std / (feature_mean_abs_shap + 1e-10)

    # Compile results
    stability_results = []
    for i, feat in enumerate(feature_names):
        if feature_mean_abs_shap[i] < NEGLIGIBLE_THRESHOLD:
            stability = 'negligible'
        elif cv[i] < 0.1:
            stability = 'high'
        elif cv[i] < 0.3:
            stability = 'medium'
        else:
            stability = 'low'

        stability_results.append({
            'feature': feat,
            'mean_abs_shap': feature_mean_abs_shap[i],
            'mean_std': feature_mean_std[i],
            'cv': cv[i],
            'stability': stability
        })

    # Sort by importance
    stability_results = sorted(stability_results, key=lambda x: -x['mean_abs_shap'])

    return stability_results, all_shap_values

# Run stability test
stability_results, all_shap_arrays = test_shap_stability(
    model, X_train, X_test, n_runs=3, background_size=500, n_test_samples=100
)

In [ ]:
# Display stability results
print("\nSHAP Explanation Stability Report")
print("=" * 75)
print(f"{'Feature':<15} {'Mean |SHAP|':>12} {'Std':>10} {'CV':>8} {'Status':>12}")
print("-" * 75)

STATUS_LABELS = {
    'high': '[PASS]',
    'medium': '[WARN]',
    'low': '[FAIL]',
    'negligible': '[ -- ]'
}

for r in stability_results:
    label = STATUS_LABELS[r['stability']]
    print(f"{r['feature']:<15} {r['mean_abs_shap']:>12.4f} {r['mean_std']:>10.4f} "
          f"{r['cv']:>8.2%} {label:>12}")

print("\nLegend:")
print("  CV (Coefficient of Variation) = Std / Mean")
print("  [PASS]  High stability: CV < 10% - SHAP values are consistent")
print("  [WARN]  Medium stability: CV 10-30% - Some variation, interpret with care")
print("  [FAIL]  Low stability: CV > 30% - High variation, may be unreliable")
print(f"  [ -- ]  Negligible: Mean |SHAP| < {NEGLIGIBLE_THRESHOLD} - too small to assess")

In [ ]:
# Visualise stability: horizontal error-bar plot
fig, ax = plt.subplots(figsize=(8, 5))

features_sorted = [r['feature'] for r in stability_results]
means = [r['mean_abs_shap'] for r in stability_results]
stds = [r['mean_std'] for r in stability_results]

y_pos = range(len(features_sorted))
ax.barh(y_pos, means, xerr=stds, align='center', color='steelblue', ecolor='coral',
        capsize=3, alpha=0.8)
ax.set_yticks(y_pos)
ax.set_yticklabels(features_sorted)
ax.invert_yaxis()
ax.set_xlabel('Mean |SHAP value|')
ax.set_title('Feature Importance with Stability Error Bars\n(Error bars = std across runs)')
plt.tight_layout()
plt.show()

print("Features with small error bars relative to their mean are stable.")
print("Features with large error bars may produce unreliable explanations.")

## 5. Generating Assurance Evidence

### 5.1 The "Failed Audit" Scenario

> *"An auditor rejects your model because Feature A and Feature B swap importance rankings randomly between runs. The explanation is unstable and therefore unsuitable as assurance evidence. Here's how to detect and address this..."*

This is why stability testing (Section 4.5) matters! Let's see what proper assurance evidence looks like.

In [ ]:
# Demonstrate instability with a very small background — the "failed audit" in action
print("Simulating a 'failed audit' with background_size=10...\n")
unstable_results, _ = test_shap_stability(
    model, X_train, X_test, n_runs=3, background_size=10, n_test_samples=50
)

print(f"\n{'Feature':<15} {'Mean |SHAP|':>12} {'CV':>8} {'Status':>8}")
print("-" * 50)
for r in unstable_results[:5]:
    label = STATUS_LABELS.get(r['stability'], '?')
    print(f"{r['feature']:<15} {r['mean_abs_shap']:>12.4f} {r['cv']:>8.2%} {label:>8}")

# Check if rankings shift
stable_ranking = [r['feature'] for r in stability_results[:3]]
unstable_ranking = [r['feature'] for r in unstable_results[:3]]
print(f"\nTop-3 with bg=500: {stable_ranking}")
print(f"Top-3 with bg=10:  {unstable_ranking}")
if stable_ranking != unstable_ranking:
    print("\n[FAIL] Rankings shifted! An auditor would reject this explanation.")
else:
    print("\nRankings held, but CV values are much higher — still a concern.")

### 5.2 What Claims Can SHAP Support?

| Claim | SHAP Support | Notes |
|-------|--------------|-------|
| ✅ "The model uses feature X in making predictions" | Strong | SHAP directly measures feature contribution |
| ✅ "Feature X has positive/negative association with outcome" | Strong | Direction of SHAP values shows this |
| ✅ "This specific prediction was driven by features X, Y, Z" | Strong | Local explanations show this exactly |
| ⚠️ "The model is fair" | Insufficient | SHAP can show *what* features are used, not whether their use is *appropriate* |
| ❌ "Feature X causes the outcome" | Not supported | SHAP shows correlation/association, not causation |

### 5.3 Generating Evidence Artefacts

An **assurance case** is a structured argument—supported by evidence—that a system has a particular property (e.g. "predictions are explainable"). It typically consists of claims, sub-claims, and evidence nodes arranged in a tree. SHAP outputs can serve as evidence nodes within such a case.

Let's generate concrete, exportable evidence for an assurance case.

In [ ]:
def get_feature_correlations(X_train, threshold=0.5):
    """
    Identify highly correlated feature pairs in the training data.
    Returns a list of dicts with feature_1, feature_2, correlation.
    """
    corr_matrix = X_train.corr()
    high_correlations = []
    for i in range(len(corr_matrix.columns)):
        for j in range(i + 1, len(corr_matrix.columns)):
            r = corr_matrix.iloc[i, j]
            if abs(r) > threshold:
                high_correlations.append({
                    'feature_1': corr_matrix.columns[i],
                    'feature_2': corr_matrix.columns[j],
                    'correlation': round(r, 3)
                })
    return high_correlations

# Quick test
corrs = get_feature_correlations(X_train)
print(f"Found {len(corrs)} highly correlated feature pairs (|r| > 0.5):")
for c in corrs:
    print(f"  {c['feature_1']} <-> {c['feature_2']}: r = {c['correlation']}")

In [ ]:
# Generate the evidence artefact — note: explainer is passed explicitly
evidence = generate_shap_evidence_artefact(
    model=model,
    explainer=explainer,
    X_train=X_train,
    X_test=X_test,
    model_id="xgb_california_housing_v1"
)

In [ ]:
def generate_shap_evidence_artefact(
    model,
    explainer,
    X_train,
    X_test,
    model_id="xgb_housing_v1",
    background_size=500,
    stability_runs=3,
    stability_threshold=0.1  # CV threshold for PASS/FAIL
):
    """
    Generate a complete SHAP evidence artefact suitable for assurance cases.

    Parameters
    ----------
    model : fitted model
    explainer : shap.TreeExplainer (passed explicitly to avoid global scope issues)
    X_train, X_test : DataFrames
    model_id : str
    background_size : int
    stability_runs : int
    stability_threshold : float - CV threshold; features above this are flagged

    Returns a dictionary that can be saved as JSON.
    Includes PASS/FAIL status for CI/CD integration.
    """
    # Model hash for reproducibility tracking
    import pickle
    model_bytes = pickle.dumps(model)
    model_hash = hashlib.sha256(model_bytes).hexdigest()[:16]

    # Model performance metrics
    y_pred_test = model.predict(X_test)
    rmse_val = np.sqrt(mean_squared_error(y_test, y_pred_test))
    r2_val = r2_score(y_test, y_pred_test)

    # Run stability analysis
    stability_results_local, _ = test_shap_stability(
        model, X_train, X_test,
        n_runs=stability_runs,
        background_size=background_size
    )

    # Check correlations using the helper function
    high_correlations = get_feature_correlations(X_train, threshold=0.5)

    # Determine overall stability and PASS/FAIL status
    top_3_stable = all(
        r['cv'] < stability_threshold
        for r in stability_results_local[:3]
        if r['stability'] != 'negligible'
    )
    active_results = [r for r in stability_results_local[:3] if r['stability'] != 'negligible']
    overall_stability = (
        'high' if all(r['stability'] == 'high' for r in active_results)
        else ('medium' if any(r['stability'] == 'high' for r in active_results)
              else 'low')
    )

    # PASS/FAIL gate for assurance
    stability_pass = top_3_stable and overall_stability in ['high', 'medium']

    # Build evidence artefact
    evidence = {
        'metadata': {
            'model_id': model_id,
            'model_hash': model_hash,
            'timestamp': datetime.now().isoformat(),
            'shap_version': shap.__version__,
            'explainer_type': 'TreeExplainer (Interventional)',
            'background_size': background_size,
            'test_samples_analysed': min(100, len(X_test))
        },
        'model_performance': {
            'rmse': round(float(rmse_val), 4),
            'r2': round(float(r2_val), 4)
        },
        'assurance_gate': {
            'status': 'PASS' if stability_pass else 'FAIL',
            'stability_threshold_cv': stability_threshold,
            'top_3_features_below_threshold': top_3_stable,
            'message': (
                'Explanations are stable and suitable for assurance evidence.'
                if stability_pass else
                'WARNING: Explanation stability below threshold. '
                'Review before using as evidence.'
            )
        },
        'feature_ranking': [
            {
                'rank': i + 1,
                'feature': r['feature'],
                'mean_abs_shap': round(r['mean_abs_shap'], 4),
                'stability': r['stability'],
                'coefficient_of_variation': round(r['cv'], 4)
            }
            for i, r in enumerate(stability_results_local)
        ],
        'stability_analysis': {
            'n_runs': stability_runs,
            'background_size': background_size,
            'overall_stability': overall_stability,
            'top_3_features_stable': all(
                r['stability'] in ['high', 'medium']
                for r in active_results
            )
        },
        'base_value': {
            'value': round(float(explainer.expected_value), 4),
            'interpretation': 'Average model prediction - SHAP values are relative to this baseline'
        },
        'limitations': [
            'SHAP values show associations, not causal effects',
            'Interventional TreeSHAP assumes feature independence when marginalising',
            f'Background dataset of {background_size} samples used for baseline',
            'Categorical features (if any) may have diluted importance due to one-hot encoding'
        ],
        'correlated_features': high_correlations if high_correlations else 'None with |r| > 0.5',
        'recommendations': []
    }

    # Add recommendations based on findings
    if high_correlations:
        evidence['recommendations'].append(
            f"Note: {len(high_correlations)} highly correlated feature pairs detected. "
            "SHAP credit may be split between these features."
        )

    if not stability_pass:
        evidence['recommendations'].append(
            "CRITICAL: Stability check failed. Consider increasing background dataset size "
            "or investigating why explanations are unstable before using as assurance evidence."
        )

    return evidence

print("Evidence artefact generator defined.")

In [ ]:
# Display the evidence artefact
print("SHAP Evidence Artefact")
print("=" * 60)
print(json.dumps(evidence, indent=2))

In [ ]:
# Save evidence artefact to file
with open('shap_evidence.json', 'w') as f:
    json.dump(evidence, f, indent=2)

print("This JSON artefact can be:")
print("  - Saved as evidence for an assurance case")
print("  - Versioned alongside model artefacts")
print("  - Referenced in audit documentation")
print("  - Compared across model versions")
print(f"\nSaved to: shap_evidence.json")

### 5.4 Assurance Case Integration

**Example Claim**: "The model's predictions are explainable at both individual and aggregate levels."

**Evidence Structure**:

```
Claim: Model predictions are explainable
├── Sub-claim: Individual predictions can be decomposed into feature contributions
│   └── Evidence: SHAP waterfall plots showing additive decomposition
├── Sub-claim: Global feature importance is stable and interpretable  
│   └── Evidence: Stability analysis showing CV < 10% for top features
└── Sub-claim: Explanations are documented with known limitations
    └── Evidence: SHAP evidence artefact with correlation warnings
```

**Confidence Qualifiers**:
- "Under the condition that background dataset has 500+ samples..."
- "Noting that features MedInc and AveOccup show correlation (r=0.65)..."
- "With the caveat that SHAP values show associations, not causal effects..."

### Linking to Assurance Goals

This tutorial generates evidence relevant to three TEA Techniques assurance goals:

- **Explainability**: SHAP provides additive, local explanations that decompose every prediction into feature contributions. The waterfall plots, beeswarm plots, and evidence artefact directly support explainability claims.

- **Fairness**: By comparing SHAP attributions across subgroups (Section 5.6), we can detect whether the model relies on features differently for different populations. This supports fairness diagnostics (though not fairness proofs).

- **Reliability**: By checking whether SHAP-identified top features align with domain expertise (Section 5.7), we can build confidence that the model has learnt meaningful relationships. Stability testing (Section 4.5) further supports reliability by showing explanations are consistent.

### 5.5 Sample Claims from TEA Techniques

The TEA Techniques entry for SHAP includes sample claims that practitioners can adapt. Here is how the evidence from this tutorial maps to those claims:

| Sample Claim | Tutorial Evidence |
|---|---|
| "The system can provide explanations for individual predictions" | Waterfall plots (Section 4.3) decompose each prediction into additive feature contributions |
| "Feature importance rankings are stable across different background samples" | Stability analysis (Section 4.5) quantifies CV per feature and provides [PASS]/[FAIL] status |
| "The model's key drivers are consistent with domain knowledge" | Reliability check (Section 5.7) compares SHAP top features against expected domain features |
| "Explanations are documented with known limitations and caveats" | Evidence artefact (Section 5.3) includes limitations, correlations, and confidence qualifiers |
| "The model does not exhibit differential reliance on features across subgroups" | Fairness analysis (Section 5.6) compares mean |SHAP| across population subgroups |

### 5.5 Limitations for Assurance

When using SHAP for assurance purposes, document these limitations:

1. **Computational, not cognitive explanations**: SHAP tells you *how the model computes*, not whether this matches human reasoning

2. **Doesn't validate correctness**: SHAP explains *what* the model does, not whether it's *right*

3. **Background dataset affects results**: Document your choice and test stability

4. **Interventional vs Path-Dependent**: Passing `data=` to TreeExplainer assumes feature independence; omitting it respects correlations captured by the tree

5. **Model-agnostic methods have approximation error**: KernelSHAP is approximate; TreeSHAP is exact for trees

6. **Categorical features (One-Hot Encoding)**: When categorical features are one-hot encoded, SHAP credit is split across dummy columns. Sum the SHAP values for all columns belonging to a single category to get the true feature importance.

7. **Adversarial vulnerability**: SHAP values can potentially be manipulated by adversarial perturbations (relevant for security-critical systems)

8. **Subgroup heterogeneity**: Global SHAP summaries (e.g. mean |SHAP|) can mask important differences between population subgroups. A feature that appears unimportant on average may be critical for a minority subgroup. Always consider disaggregated analysis alongside global summaries.

**Interpreting the reliability check:**

High alignment between SHAP-identified top features and domain expectations increases confidence that the model has learnt meaningful relationships rather than spurious correlations.

**Supported claim:** *"The model's most influential features, as identified by SHAP, are consistent with domain expertise about California housing price drivers."*

**Connection to Limitation 4:** If you used Path-Dependent TreeSHAP instead of Interventional, the feature rankings might differ due to how correlated features are handled. Always document which mode was used.

In [ ]:
# Reliability check: do top SHAP features match domain expectations?

# Domain knowledge: these features SHOULD be important for house prices
EXPECTED_TOP = {'MedInc', 'Latitude', 'Longitude', 'AveRooms', 'HouseAge'}

# Get actual top features from SHAP
mean_abs_shap = np.mean(np.abs(shap_values.values), axis=0)
feature_importance = sorted(
    zip(X_test.columns, mean_abs_shap),
    key=lambda x: -x[1]
)

# Top-5 actual features
actual_top = {f[0] for f in feature_importance[:5]}
overlap = EXPECTED_TOP & actual_top
missing = EXPECTED_TOP - actual_top
unexpected = actual_top - EXPECTED_TOP

print("Reliability Check: Domain Alignment")
print("=" * 55)
print(f"\nExpected top features: {sorted(EXPECTED_TOP)}")
print(f"Actual top features:  {sorted(actual_top)}")
print(f"\nOverlap:    {sorted(overlap) if overlap else 'None'}")
print(f"Missing:    {sorted(missing) if missing else 'None'}")
print(f"Unexpected: {sorted(unexpected) if unexpected else 'None'}")

alignment = len(overlap) / len(EXPECTED_TOP)
status = '[PASS]' if alignment >= 0.6 else '[WARN]' if alignment >= 0.4 else '[FAIL]'
print(f"\nAlignment score: {alignment:.0%} ({len(overlap)}/{len(EXPECTED_TOP)}) {status}")

if missing:
    print(f"\nNote: Features {sorted(missing)} were expected to be important but are not")
    print("in the top 5. This may warrant domain expert review.")

### 5.7 SHAP for Reliability Assurance

The TEA Techniques framework also lists SHAP under **Reliability** because it can verify that the model's most important features align with domain expertise. If the model relies heavily on features that domain experts consider irrelevant (or ignores features they consider essential), this raises reliability concerns.

**Approach:** Define an expected set of top features based on domain knowledge, then check whether the model's actual top features (by mean |SHAP|) overlap.

**Interpreting the fairness check:**

Large differences in mean |SHAP| between subgroups indicate the model relies on certain features more heavily for one group than the other. This is not necessarily unfair — it may reflect genuine differences in the data — but it warrants investigation.

**Supported claim:** *"We have examined whether the model's reliance on individual features differs systematically across population subgroups, using SHAP-based attribution comparison."*

**Important caveat:** This is a diagnostic, not a fairness proof. It shows *differential reliance*, which may or may not constitute unfairness depending on the domain and regulatory context. See Limitation 8 (subgroup heterogeneity) above.

In [ ]:
# Fairness analysis: compare SHAP attributions across subgroups
# Split by MedInc median as a proxy for socioeconomic subgroups
medinc_median = X_test['MedInc'].median()
low_income_mask = X_test['MedInc'] <= medinc_median
high_income_mask = ~low_income_mask

shap_abs = np.abs(shap_values.values)

low_income_mean = shap_abs[low_income_mask].mean(axis=0)
high_income_mean = shap_abs[high_income_mask].mean(axis=0)

print("Fairness Check: Mean |SHAP| by Subgroup")
print("=" * 70)
print(f"{'Feature':<15} {'Low-Income':>12} {'High-Income':>12} {'Difference':>12}")
print("-" * 70)

for i, feat in enumerate(X_test.columns):
    diff = abs(high_income_mean[i] - low_income_mean[i])
    flag = " <--" if diff > 0.05 else ""
    print(f"{feat:<15} {low_income_mean[i]:>12.4f} {high_income_mean[i]:>12.4f} "
          f"{diff:>12.4f}{flag}")

print("\nFeatures flagged (<--) have > 0.05 difference in mean |SHAP| between groups.")
print("This suggests the model weights these features differently across subgroups.")

### 5.6 SHAP for Fairness Assurance

The TEA Techniques framework lists SHAP under **Fairness** because it can reveal whether a model treats different population subgroups differently. By comparing SHAP attributions across groups, we can detect differential model behaviour that may indicate unfairness.

**Approach:** Split the test set into subgroups and compare mean |SHAP| values per feature across groups. Large differences suggest the model relies on features differently for different populations.

## 6. Going Further

### Other SHAP Explainers

| Explainer | Best For | Speed | Exactness |
|-----------|----------|-------|--------|
| TreeExplainer | Tree models (XGBoost, RF, etc.) | Fast | Exact |
| DeepExplainer | Neural networks (TensorFlow/PyTorch) | Medium | Approximate |
| GradientExplainer | Neural networks | Medium | Approximate |
| KernelExplainer | Any model | Slow | Approximate |
| LinearExplainer | Linear models | Fast | Exact |

### Advanced Topics

- **SHAP interaction values**: Measure how pairs of features interact
- **Text explanations**: Explain NLP model predictions word-by-word
- **Image explanations**: Highlight important regions in image classifications

### Related Techniques

- **[LIME](https://alan-turing-institute.github.io/tea-techniques/techniques/local-interpretable-model-agnostic-explanations)**: Another local explanation method; fits a simple model around each prediction
- **Integrated Gradients**: Gradient-based explanations for neural networks
- **[Permutation Importance](https://scikit-learn.org/stable/modules/permutation_importance.html)**: Global importance via feature shuffling; complements SHAP's local explanations
- **[Partial Dependence Plots (PDP)](https://scikit-learn.org/stable/modules/partial_dependence.html)**: Visualise marginal effect of a feature; pairs well with SHAP dependence plots
- **[Shapash](https://github.com/MAIF/shapash)**: Higher-level SHAP wrapper with interactive dashboards and report generation

### Additional Resources

- [SHAP Documentation](https://shap.readthedocs.io/)
- [Original SHAP Paper (Lundberg & Lee, 2017)](https://arxiv.org/abs/1705.07874)
- [Yuan et al. (2022) - SHAP Stability Study](http://arxiv.org/abs/2204.11351)
- [TEA Techniques: SHAP](https://alan-turing-institute.github.io/tea-techniques/techniques/shapley-additive-explanations)

**Key differences between KernelSHAP and TreeSHAP:**

| | TreeSHAP | KernelSHAP |
|---|----------|------------|
| **Models** | Tree-based only | Any model (black-box) |
| **Speed** | Fast (polynomial) | Slow (requires many model evaluations) |
| **Exactness** | Exact Shapley values | Approximate (sampling-based) |
| **Background size** | 500+ recommended | 50-100 often sufficient (cost-constrained) |

**For assurance:** KernelSHAP's approximation introduces additional uncertainty. When using it as evidence, document the background size, number of samples explained, and consider running multiple seeds to assess estimation variance.

In [ ]:
# KernelSHAP demo: treat a RandomForest as a black box
from sklearn.ensemble import RandomForestRegressor

# Train a RandomForest (could be any model)
rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

# KernelExplainer needs a smaller background (it's much slower than TreeSHAP)
kernel_background = shap.sample(X_train, 50, random_state=42)
kernel_explainer = shap.KernelExplainer(rf_model.predict, kernel_background)

# Explain just 5 samples (KernelSHAP is slow)
kernel_samples = X_test.iloc[:5]
print("Computing KernelSHAP values for 5 samples (this may take a moment)...")
kernel_shap_values = kernel_explainer.shap_values(kernel_samples)

# Show waterfall for first sample
kernel_explanation = shap.Explanation(
    values=kernel_shap_values[0],
    base_values=kernel_explainer.expected_value,
    data=kernel_samples.iloc[0].values,
    feature_names=list(X_test.columns)
)
shap.plots.waterfall(kernel_explanation, show=False)
plt.title("KernelSHAP: Black-Box Random Forest Explanation", fontsize=10)
plt.tight_layout()
plt.show()

print(f"\nKernelSHAP base value: ${kernel_explainer.expected_value * 100000:,.0f}")
print(f"Prediction: ${rf_model.predict(kernel_samples.iloc[[0]])[0] * 100000:,.0f}")

### 6.1 KernelSHAP: Explaining Any Black-Box Model

TreeSHAP only works with tree-based models. For any other model (neural networks, SVMs, ensembles), **KernelSHAP** provides model-agnostic approximate SHAP values. It treats the model as a black box and uses weighted linear regression to estimate Shapley values.

Let's demonstrate with a Random Forest treated as a black box:

## 7. Reflective Questions

Before finishing, consider these questions:

### Conceptual Understanding

1. **If two features are highly correlated, what happens to their SHAP values? How would you address this in an assurance context?**

<details>
<summary>Discussion notes</summary>

SHAP credit gets "split" between correlated features—each receives a share of the importance that might otherwise go entirely to one. This is mathematically correct under the Shapley axioms, but can mislead stakeholders who expect a single "root cause." For assurance, document the correlation, present grouped feature importances, and note that the sum of correlated features' SHAP values is more meaningful than either alone.
</details>

2. **Why is the base value important for interpreting SHAP values?**

<details>
<summary>Discussion notes</summary>

The base value is the anchor for all SHAP values. A SHAP value of +$50,000 means "$50,000 more than the average prediction," not "$50,000 in absolute terms." Without knowing the base value, you cannot convert SHAP values into actual prediction components. Misunderstanding this leads to incorrect assurance claims about which features "cause" certain outcomes.
</details>

### Application Scenarios

3. **A stakeholder asks why a loan was denied. How would you use SHAP to answer, and what caveats would you communicate?**

<details>
<summary>Discussion notes</summary>

Generate a waterfall plot for the specific application. Identify the top features pushing the prediction below the approval threshold. Communicate that SHAP shows which features influenced the model's decision, but: (a) it doesn't prove the features *caused* the denial, (b) it doesn't say whether the model's use of those features is fair or appropriate, and (c) correlated features may share credit.
</details>

4. **You're auditing a model and notice SHAP values change significantly with different background datasets. What does this tell you, and what would you recommend?**

<details>
<summary>Discussion notes</summary>

Instability indicates the explanations are sensitive to the choice of baseline. This undermines their reliability as assurance evidence. Recommend: (a) increase the background dataset size (500+ samples), (b) run multiple seeds and report variance, (c) if instability persists, investigate whether the model itself is overfit or whether feature correlations are causing the issue. Do not use unstable explanations as audit evidence without documenting the uncertainty.
</details>

### Critical Thinking

5. **Can SHAP prove a model is "fair"? What additional evidence would you need?**

<details>
<summary>Discussion notes</summary>

No. SHAP can show *what features* the model relies on and whether protected attributes (or their proxies) have high importance, but it cannot determine whether the model's use of those features is *appropriate*. Fairness requires domain-specific analysis: statistical parity, equalised odds, or other fairness metrics applied to the model's predictions across protected groups. SHAP is a useful diagnostic, not a fairness proof.
</details>

6. **An engineer says "SHAP proves that income causes higher prices in our model." How would you respond?**

<details>
<summary>Discussion notes</summary>

SHAP shows that the model *associates* higher income with higher predicted prices—it does not prove causation. The model could be picking up on confounders correlated with income. True causal claims require causal inference methods (e.g. instrumental variables, randomised experiments). For assurance documentation, always use language like "the model associates" or "the model relies on" rather than "causes."
</details>

7. **When should SHAP NOT be used as the primary explainability technique?**

<details>
<summary>Discussion notes</summary>

SHAP may be inappropriate when: (a) the model is inherently interpretable (e.g. small decision tree, logistic regression with few features)—direct inspection is simpler and more reliable; (b) real-time explanations are needed and SHAP computation is too slow; (c) the feature space is very high-dimensional (thousands of features) making SHAP plots unreadable; (d) causal explanations are required rather than associative ones; (e) the audience needs contrastive explanations ("why this and not that?") which SHAP doesn't directly provide.
</details>

---

## Summary

In this tutorial, you learnt:

- **What SHAP is**: A method to explain predictions by computing each feature's contribution based on Shapley values from game theory

- **How to apply it**: Using TreeExplainer for tree models and KernelExplainer for black-box models, with appropriate background datasets (500+ samples)

- **How to interpret results**: Waterfall plots for local explanations, beeswarm plots for global patterns, dependence plots for interactions

- **Common pitfalls**: Small background datasets, correlated features, confusing correlation with causation

- **Assurance applications**: Generating evidence artefacts, testing stability, documenting limitations

- **Fairness assurance**: Comparing SHAP attributions across subgroups to detect differential model behaviour

- **Reliability assurance**: Verifying that top model features align with domain knowledge

**Key Takeaways for Assurance**:
1. Always test explanation stability before using SHAP as evidence
2. Document correlated features and their impact on credit splitting
3. Be precise: SHAP shows associations, not causal effects
4. Generate structured evidence artefacts that can be versioned and audited
5. Consider subgroup heterogeneity — global summaries can mask important differences

**Next Steps**:
- Apply SHAP to your own models
- Explore the [LIME tutorial](https://alan-turing-institute.github.io/tea-techniques/tutorials/notebooks/lime/) for comparison
- Read the original paper: Lundberg & Lee (2017)

---

*This tutorial is part of the [TEA Techniques](https://alan-turing-institute.github.io/tea-techniques/) project — helping practitioners generate assurance evidence for AI systems.*